In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import pandas as pd
import numpy as np

for dirname, _, filenames in os.walk(
    "/kaggle/input/datasets/bilalahmad302/ai-social-media-and-students-performance"
):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/bilalahmad302/ai-social-media-and-students-performance/AI_SocialMedia_Student_Dataset.csv


In [ ]:
import pandas as pd

file_path = "/kaggle/input/datasets/bilalahmad302/ai-social-media-and-students-performance/AI_SocialMedia_Student_Dataset.csv"

df = pd.read_csv(file_path)

print("Dataset Title: AI Social Media and Students Performance")
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst Five Rows:")
display(df.head())

Dataset Title: AI Social Media and Students Performance
Number of Rows: 16000
Number of Columns: 10

Column Names:
['Student_ID', 'Age', 'Gender', 'Education_Level', 'Daily_Social_Media_Hours', 'Daily_AI_Tool_Usage_Hours', 'Sleep_Hours', 'Physical_Activity_Hours', 'Mental_Health_Score', 'Physical_Health_Score']

First Five Rows:


,Student_ID,Age,Gender,Education_Level,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score
0,STU_00001,19,Male,College,5.32,1.21,7.30,1.41,74.06,98.90
1,STU_00002,16,Non-binary,High School,5.21,3.89,7.81,2.48,76.13,89.94
2,STU_00003,25,Male,University,3.61,2.65,6.34,0.06,65.01,76.75
3,STU_00004,23,Male,University,2.93,1.43,6.12,1.37,74.67,87.38
4,STU_00005,20,Non-binary,College,6.52,1.64,5.23,1.14,67.74,88.04


Dataset Title: AI Social Media and Students Performance
Main Features: The dataset contains information related to social media and students' academic performance. The exact features are listed in the dataset columns.
Objective: The objective of this dataset is to explore the relationship between students' social media usage and their academic performance. It can help us examine whether social media habits are associated with differences in students' academic results.



In [ ]:

import pandas as pd
import numpy as np
import os

# ==========================================
# 1. LOAD DATASET
# ==========================================

file_path = "/kaggle/input/datasets/bilalahmad302/ai-social-media-and-students-performance/AI_SocialMedia_Student_Dataset.csv"

df = pd.read_csv(file_path)

print("DATASET LOADED SUCCESSFULLY")
print("Original dataset shape:", df.shape)
display(df.head())

# Keep the original data unchanged
df_clean = df.copy()

# ==========================================
# 2. CHECK MISSING VALUES
# ==========================================

print("\n--- MISSING VALUES BEFORE CLEANING ---")

missing_report = pd.DataFrame({
    "Missing Count": df_clean.isnull().sum(),
    "Missing Percentage": (
        df_clean.isnull().sum() / len(df_clean) * 100
    ).round(2)
})

display(missing_report)
print("Total missing values:", df_clean.isnull().sum().sum())

# ==========================================
# 3. REMOVE DUPLICATE RECORDS
# ==========================================

print("\n--- DUPLICATE RECORDS ---")

print("Duplicates before removal:", df_clean.duplicated().sum())

df_clean = df_clean.drop_duplicates().copy()

print("Duplicates after removal:", df_clean.duplicated().sum())

# ==========================================
# 4. CORRECT INCONSISTENT CATEGORIES
# ==========================================

print("\n--- CATEGORICAL VALUES BEFORE CLEANING ---")

categorical_cols = df_clean.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in categorical_cols:
    print(f"\n{col}:")
    print(df_clean[col].value_counts(dropna=False).head(20))

# Remove leading/trailing and repeated spaces
for col in categorical_cols:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("\nUnnecessary spaces removed from text columns.")

# ==========================================
# 5. HANDLE MISSING VALUES
# ==========================================

print("\n--- HANDLING MISSING VALUES ---")

numeric_cols = df_clean.select_dtypes(
    include=np.number
).columns.tolist()

categorical_cols = df_clean.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

# Fill numerical missing values with median
for col in numeric_cols:
    if df_clean[col].isnull().any():
        median_value = df_clean[col].median()

        if pd.notna(median_value):
            df_clean[col] = df_clean[col].fillna(median_value)

# Fill categorical missing values with mode
for col in categorical_cols:
    if df_clean[col].isnull().any():
        mode_values = df_clean[col].mode()

        if not mode_values.empty:
            df_clean[col] = df_clean[col].fillna(
                mode_values.iloc[0]
            )

print("Missing values after cleaning:")
display(df_clean.isnull().sum())

print("Total remaining missing values:",
      df_clean.isnull().sum().sum())

# ==========================================
# 6. CHECK DATA TYPES
# ==========================================

print("\n--- DATA TYPES ---")
df_clean.info()

print("\nData type correction must be based on column meaning.")
print("Review the data types above before converting columns.")

# ==========================================
# 7. DETECT POTENTIAL OUTLIERS USING IQR
# ==========================================

print("\n--- OUTLIER DETECTION ---")

numeric_cols = df_clean.select_dtypes(
    include=np.number
).columns.tolist()

outlier_results = []

for col in numeric_cols:
    values = df_clean[col].dropna()

    if values.empty:
        continue

    Q1 = values.quantile(0.25)
    Q3 = values.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    count = (
        (values < lower_bound) |
        (values > upper_bound)
    ).sum()

    outlier_results.append({
        "Column": col,
        "Lower Bound": round(lower_bound, 2),
        "Upper Bound": round(upper_bound, 2),
        "Potential Outliers": int(count)
    })

outlier_report = pd.DataFrame(outlier_results)
display(outlier_report)

print("Outliers are flagged for investigation, not removed automatically.")

# ==========================================
# 8. FINAL VERIFICATION
# ==========================================

print("\n--- FINAL DATA QUALITY REPORT ---")

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Remaining missing values:",
      df_clean.isnull().sum().sum())
print("Remaining duplicate rows:",
      df_clean.duplicated().sum())

display(df_clean.head())

# ==========================================
# 9. SAVE CLEANED DATASET
# ==========================================

output_path = "/kaggle/working/AI_SocialMedia_Student_Cleaned.csv"

df_clean.to_csv(output_path, index=False)

print("\nCleaned dataset saved successfully!")
print("File location:", output_path)


DATASET LOADED SUCCESSFULLY
Original dataset shape: (16000, 10)


,Student_ID,Age,Gender,Education_Level,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score
0,STU_00001,19,Male,College,5.32,1.21,7.30,1.41,74.06,98.90
1,STU_00002,16,Non-binary,High School,5.21,3.89,7.81,2.48,76.13,89.94
2,STU_00003,25,Male,University,3.61,2.65,6.34,0.06,65.01,76.75
3,STU_00004,23,Male,University,2.93,1.43,6.12,1.37,74.67,87.38
4,STU_00005,20,Non-binary,College,6.52,1.64,5.23,1.14,67.74,88.04



--- MISSING VALUES BEFORE CLEANING ---


,Missing Count,Missing Percentage
Student_ID,0,0.0
Age,0,0.0
Gender,0,0.0
Education_Level,0,0.0
Daily_Social_Media_Hours,0,0.0
Daily_AI_Tool_Usage_Hours,0,0.0
Sleep_Hours,0,0.0
Physical_Activity_Hours,0,0.0
Mental_Health_Score,0,0.0
Physical_Health_Score,0,0.0


Total missing values: 0

--- DUPLICATE RECORDS ---
Duplicates before removal: 0
Duplicates after removal: 0

--- CATEGORICAL VALUES BEFORE CLEANING ---

Student_ID:
Student_ID
STU_15984    1
STU_15983    1
STU_15982    1
STU_15981    1
STU_15980    1
STU_15979    1
STU_15978    1
STU_15977    1
STU_15976    1
STU_15975    1
STU_15974    1
STU_15973    1
STU_15972    1
STU_15971    1
STU_15970    1
STU_15969    1
STU_00032    1
STU_00031    1
STU_00030    1
STU_00029    1
Name: count, dtype: int64

Gender:
Gender
Male          7744
Female        7628
Non-binary     628
Name: count, dtype: int64

Education_Level:
Education_Level
High School    6083
University     5027
College        4890
Name: count, dtype: int64

Unnecessary spaces removed from text columns.

--- HANDLING MISSING VALUES ---
Missing values after cleaning:


...

Total remaining missing values: 0

--- DATA TYPES ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Student_ID                 16000 non-null  string 
 1   Age                        16000 non-null  int64  
 2   Gender                     16000 non-null  string 
 3   Education_Level            16000 non-null  string 
 4   Daily_Social_Media_Hours   16000 non-null  float64
 5   Daily_AI_Tool_Usage_Hours  16000 non-null  float64
 6   Sleep_Hours                16000 non-null  float64
 7   Physical_Activity_Hours    16000 non-null  float64
 8   Mental_Health_Score        16000 non-null  float64
 9   Physical_Health_Score      16000 non-null  float64
dtypes: float64(6), int64(1), string(3)
memory usage: 1.2 MB

Data type correction must be based on column meaning.
Review the data types above before converting col

,Column,Lower Bound,Upper Bound,Potential Outliers
0,Age,7.00,31.00,0
1,Daily_Social_Media_Hours,-2.17,11.19,49
2,Daily_AI_Tool_Usage_Hours,-2.35,7.41,51
3,Sleep_Hours,3.08,10.01,105
4,Physical_Activity_Hours,-2.16,4.43,55
5,Mental_Health_Score,47.58,98.73,185
6,Physical_Health_Score,58.04,120.65,63


Outliers are flagged for investigation, not removed automatically.

--- FINAL DATA QUALITY REPORT ---
Original shape: (16000, 10)
Cleaned shape: (16000, 10)
Remaining missing values: 0
Remaining duplicate rows: 0


,Student_ID,Age,Gender,Education_Level,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score
0,STU_00001,19,Male,College,5.32,1.21,7.30,1.41,74.06,98.90
1,STU_00002,16,Non-binary,High School,5.21,3.89,7.81,2.48,76.13,89.94
2,STU_00003,25,Male,University,3.61,2.65,6.34,0.06,65.01,76.75
3,STU_00004,23,Male,University,2.93,1.43,6.12,1.37,74.67,87.38
4,STU_00005,20,Non-binary,College,6.52,1.64,5.23,1.14,67.74,88.04



Cleaned dataset saved successfully!
File location: /kaggle/working/AI_SocialMedia_Student_Cleaned.csv


### Data Cleaning Decisions

1. **Missing Values:** I checked missing values in each column. Numerical missing values were filled using the median, while categorical missing values were filled using the mode.

2. **Duplicate Records:** I identified and removed exact duplicate rows to prevent repeated observations from affecting the analysis.

3. **Inconsistent Categories:** I removed unnecessary spaces from text values. Other category changes should be made only when the values are confirmed to mean the same thing.

4. **Data Types:** I inspected the data types and will correct any inappropriate types according to the meaning of each column.

5. **Outliers:** I used the IQR method to identify potential outliers. I will investigate unusual values before deciding whether correction or removal is appropriate.

6. **Final Verification:** I checked missing values and duplicate records again and saved the cleaned dataset as a CSV file.

In [ ]:
# Display all column names
print("Dataset columns:")
print(df_clean.columns.tolist())

# Check the first five rows
display(df_clean.head())

# Check data types
display(df_clean.dtypes)

Dataset columns:
['Student_ID', 'Age', 'Gender', 'Education_Level', 'Daily_Social_Media_Hours', 'Daily_AI_Tool_Usage_Hours', 'Sleep_Hours', 'Physical_Activity_Hours', 'Mental_Health_Score', 'Physical_Health_Score']


,Student_ID,Age,Gender,Education_Level,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score
0,STU_00001,19,Male,College,5.32,1.21,7.30,1.41,74.06,98.90
1,STU_00002,16,Non-binary,High School,5.21,3.89,7.81,2.48,76.13,89.94
2,STU_00003,25,Male,University,3.61,2.65,6.34,0.06,65.01,76.75
3,STU_00004,23,Male,University,2.93,1.43,6.12,1.37,74.67,87.38
4,STU_00005,20,Non-binary,College,6.52,1.64,5.23,1.14,67.74,88.04


Student_ID                   string[python]
Age                                   int64
Gender                       string[python]
Education_Level              string[python]
Daily_Social_Media_Hours            float64
Daily_AI_Tool_Usage_Hours           float64
Sleep_Hours                         float64
Physical_Activity_Hours             float64
Mental_Health_Score                 float64
Physical_Health_Score               float64
dtype: object

In [ ]:
# Identify variable types in our dataset

numerical_cols = [
    'Age',
    'Daily_Social_Media_Hours',
    'Daily_AI_Tool_Usage_Hours',
    'Sleep_Hours',
    'Physical_Activity_Hours',
    'Mental_Health_Score',
    'Physical_Health_Score'
]

categorical_cols = ['Gender', 'Education_Level']
identifier_cols = ['Student_ID']

print("Numerical variables:", numerical_cols)
print("Categorical variables:", categorical_cols)
print("Identifier variables:", identifier_cols)
print("Date/time variables: None")

Numerical variables: ['Age', 'Daily_Social_Media_Hours', 'Daily_AI_Tool_Usage_Hours', 'Sleep_Hours', 'Physical_Activity_Hours', 'Mental_Health_Score', 'Physical_Health_Score']
Categorical variables: ['Gender', 'Education_Level']
Identifier variables: ['Student_ID']
Date/time variables: None


In [ ]:
print("Gender categories:")
print(df_clean['Gender'].value_counts(dropna=False))

print("\nEducation levels:")
print(df_clean['Education_Level'].value_counts(dropna=False))

print("\nNumber of unique values in each column:")
display(df_clean.nunique())

Gender categories:
Gender
Male          7744
Female        7628
Non-binary     628
Name: count, dtype: Int64

Education levels:
Education_Level
High School    6083
University     5027
College        4890
Name: count, dtype: Int64

Number of unique values in each column:


Student_ID                   16000
Age                             13
Gender                           3
Education_Level                  3
Daily_Social_Media_Hours      1106
Daily_AI_Tool_Usage_Hours      762
Sleep_Hours                    752
Physical_Activity_Hours        462
Mental_Health_Score           3743
Physical_Health_Score         3433
dtype: int64

There is no binary variable among these two categorical columns.
We will use:
- One-hot encoding for Gender, because its categories have no natural order.
- Ordinal encoding for Education_Level, because education levels have an order.


In [ ]:
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd

# Make a copy for preprocessing
df_preprocessed = df_clean.copy()

# One-hot encode Gender
df_preprocessed = pd.get_dummies(
    df_preprocessed,
    columns=['Gender'],
    dtype=int
)

# Ordinal encode Education_Level
education_order = [['High School', 'College', 'University']]

encoder = OrdinalEncoder(categories=education_order)

df_preprocessed['Education_Level'] = encoder.fit_transform(
    df_preprocessed[['Education_Level']]
).astype(int)

print("Encoding completed!")
display(df_preprocessed.head())
print("New columns:", df_preprocessed.columns.tolist())

Encoding completed!


,Student_ID,Age,Education_Level,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score,Gender_Female,Gender_Male,Gender_Non-binary
0,STU_00001,19,1,5.32,1.21,7.30,1.41,74.06,98.90,0,1,0
1,STU_00002,16,0,5.21,3.89,7.81,2.48,76.13,89.94,0,0,1
2,STU_00003,25,2,3.61,2.65,6.34,0.06,65.01,76.75,0,1,0
3,STU_00004,23,2,2.93,1.43,6.12,1.37,74.67,87.38,0,1,0
4,STU_00005,20,1,6.52,1.64,5.23,1.14,67.74,88.04,0,0,1


New columns: ['Student_ID', 'Age', 'Education_Level', 'Daily_Social_Media_Hours', 'Daily_AI_Tool_Usage_Hours', 'Sleep_Hours', 'Physical_Activity_Hours', 'Mental_Health_Score', 'Physical_Health_Score', 'Gender_Female', 'Gender_Male', 'Gender_Non-binary']


Why this encoding? 
One-hot encoding avoids suggesting that one gender is higher than another. Ordinal encoding represents education levels as High School = 0, College = 1, and 
University = 2.

In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols = [
    'Age',
    'Daily_Social_Media_Hours',
    'Daily_AI_Tool_Usage_Hours',
    'Sleep_Hours',
    'Physical_Activity_Hours',
    'Mental_Health_Score',
    'Physical_Health_Score'
]

scaler = StandardScaler()

df_preprocessed[scale_cols] = scaler.fit_transform(
    df_preprocessed[scale_cols]
)

print("Standardization completed!")
display(df_preprocessed.head())

Standardization completed!


,Student_ID,Age,Education_Level,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score,Gender_Female,Gender_Male,Gender_Non-binary
0,STU_00001,-0.010292,1,0.323853,-0.822131,0.593974,0.155634,0.169485,1.122674,0,1,0
1,STU_00002,-0.807072,0,0.278076,0.773750,0.995398,1.185868,0.393401,0.198079,0,0,1
2,STU_00003,1.583269,2,-0.387767,0.035357,-0.161649,-1.144194,-0.809473,-1.163015,0,1,0
3,STU_00004,1.052082,2,-0.670750,-0.691126,-0.334813,0.117120,0.235470,-0.066090,0,1,0
4,STU_00005,0.255302,1,0.823235,-0.566075,-1.035338,-0.104332,-0.514163,0.002016,0,0,1


standardization worked successfully. The numerical values are now scaled around a mean of 0, and the gender categories have been one-hot encoded.

In [ ]:
print("Missing values in preprocessed dataset:")
display(df_preprocessed.isnull().sum())

print(
    "Total missing values:",
    df_preprocessed.isnull().sum().sum()
)

Missing values in preprocessed dataset:


Student_ID                   0
Age                          0
Education_Level              0
Daily_Social_Media_Hours     0
Daily_AI_Tool_Usage_Hours    0
Sleep_Hours                  0
Physical_Activity_Hours      0
Mental_Health_Score          0
Physical_Health_Score        0
Gender_Female                0
Gender_Male                  0
Gender_Non-binary            0
dtype: int64

Total missing values: 0


In [ ]:
skewness = df_clean.select_dtypes(include='number').skew()

print("Skewness of numerical columns:")
display(skewness)

Skewness of numerical columns:


Age                         -0.011042
Daily_Social_Media_Hours     0.156575
Daily_AI_Tool_Usage_Hours    0.338173
Sleep_Hours                  0.014566
Physical_Activity_Hours      0.618817
Mental_Health_Score         -0.682640
Physical_Health_Score       -0.749444
dtype: float64

In [ ]:
df_preprocessed['Total_Daily_Usage_Hours'] = (
    df_clean['Daily_Social_Media_Hours'] +
    df_clean['Daily_AI_Tool_Usage_Hours']
)

print("New feature created successfully!")
display(
    df_preprocessed[
        ['Student_ID', 'Total_Daily_Usage_Hours']
    ].head()
)

New feature created successfully!


,Student_ID,Total_Daily_Usage_Hours
0,STU_00001,6.53
1,STU_00002,9.10
2,STU_00003,6.26
3,STU_00004,4.36
4,STU_00005,8.16


In [ ]:
print("Final preprocessed dataset shape:", df_preprocessed.shape)
print("Total missing values:", df_preprocessed.isnull().sum().sum())
print("Duplicate rows:", df_preprocessed.duplicated().sum())

display(df_preprocessed.head())

df_preprocessed.to_csv(
    "/kaggle/working/AI_SocialMedia_Student_Preprocessed.csv",
    index=False
)

print("Preprocessed dataset saved successfully!")

Final preprocessed dataset shape: (16000, 13)
Total missing values: 0
Duplicate rows: 0


,Student_ID,Age,Education_Level,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score,Gender_Female,Gender_Male,Gender_Non-binary,Total_Daily_Usage_Hours
0,STU_00001,-0.010292,1,0.323853,-0.822131,0.593974,0.155634,0.169485,1.122674,0,1,0,6.53
1,STU_00002,-0.807072,0,0.278076,0.773750,0.995398,1.185868,0.393401,0.198079,0,0,1,9.10
2,STU_00003,1.583269,2,-0.387767,0.035357,-0.161649,-1.144194,-0.809473,-1.163015,0,1,0,6.26
3,STU_00004,1.052082,2,-0.670750,-0.691126,-0.334813,0.117120,0.235470,-0.066090,0,1,0,4.36
4,STU_00005,0.255302,1,0.823235,-0.566075,-1.035338,-0.104332,-0.514163,0.002016,0,0,1,8.16


Preprocessed dataset saved successfully!
